# SkinVision AI MobileNetV2

**Judul proyek:** Pengembangan Aplikasi Diagnosis Penyakit Kulit Menggunakan Artificial Intelligence dan Computer Vision

Notebook ini digunakan untuk melatih model klasifikasi gambar kulit menggunakan **Transfer Learning MobileNetV2**.

Dataset yang digunakan berisi 7 class:

1. Acne
2. Blackheads
3. Dark Spots
4. Dry Skin
5. Normal Skin
6. Oily Skin
7. Wrinkles

Output akhir notebook:

- `skin_disease_model.h5`
- `class_names.txt`

Dua file tersebut dapat dipindahkan ke folder web Flask: `skinvision-ai-web/model/`.

## BAB 1. Pendahuluan

Aplikasi SkinVision AI dibuat untuk membantu pengguna melakukan klasifikasi awal kondisi kulit berdasarkan gambar. Sistem ini memakai Artificial Intelligence dan Computer Vision. Model menerima input gambar kulit, melakukan preprocessing, lalu menghasilkan prediksi class beserta tingkat confidence.

Aplikasi ini hanya digunakan untuk demo proyek UAS. Hasil prediksi tidak menggantikan pemeriksaan dokter.

Tujuan proyek:

- Membuat model AI untuk klasifikasi gambar kulit.
- Menggunakan Transfer Learning MobileNetV2 agar model lebih kuat dibanding CNN sederhana.
- Menyimpan model ke format `.h5` agar dapat digunakan pada web Flask.
- Menampilkan hasil prediksi, confidence score, dan status awal pada aplikasi web.

## BAB 2. Setup Environment dan Import Library

Jalankan cell ini terlebih dahulu. Pastikan runtime Google Colab memakai GPU.

Cara aktifkan GPU:

Runtime → Change runtime type → Hardware accelerator → GPU → Save

In [ ]:
import os
import zipfile
import random
import shutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras import layers, models
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing import image

from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

print("TensorFlow version:", tf.__version__)
print("GPU tersedia:", tf.config.list_physical_devices('GPU'))

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

## BAB 3. Upload dan Ekstraksi Dataset

Upload file dataset kamu, yaitu `archive (4).zip`.

Struktur dataset setelah diekstrak harus mengarah ke folder yang berisi class gambar kulit.

In [ ]:
from google.colab import files

uploaded = files.upload()
zip_name = list(uploaded.keys())[0]
print("File yang diupload:", zip_name)

In [ ]:
extract_path = "/content/skinvision_dataset"

if os.path.exists(extract_path):
    shutil.rmtree(extract_path)

os.makedirs(extract_path, exist_ok=True)

with zipfile.ZipFile(zip_name, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print("Dataset berhasil diekstrak ke:", extract_path)

for root, dirs, files_ in os.walk(extract_path):
    level = root.replace(extract_path, '').count(os.sep)
    indent = ' ' * 2 * level
    print(f"{indent}{os.path.basename(root)}/")
    if level >= 2:
        dirs[:] = []

### Deteksi Otomatis Folder Dataset

Cell ini mencari folder yang berisi subfolder class seperti `Acne`, `Blackheads`, dan class lainnya.

In [ ]:
def find_dataset_dir(base_path):
    expected = {"Acne", "Blackheads", "Dark Spots", "Dry Skin", "Normal Skin", "Oily Skin", "Wrinkles"}
    candidates = []

    for root, dirs, files_ in os.walk(base_path):
        dir_set = set(dirs)
        score = len(expected.intersection(dir_set))
        if score >= 3:
            candidates.append((score, root, dirs))

    if not candidates:
        raise FileNotFoundError("Folder dataset tidak ditemukan. Pastikan ZIP berisi folder class gambar.")

    candidates.sort(reverse=True, key=lambda x: x[0])
    return candidates[0][1]

dataset_dir = find_dataset_dir(extract_path)
print("Folder dataset yang digunakan:", dataset_dir)
print("Class yang ditemukan:", os.listdir(dataset_dir))

## BAB 4. Eksplorasi Dataset

Bagian ini menampilkan jumlah gambar pada setiap class. Ini penting untuk melihat apakah dataset seimbang atau tidak.

In [ ]:
class_counts = []

for class_name in sorted(os.listdir(dataset_dir)):
    class_path = os.path.join(dataset_dir, class_name)
    if os.path.isdir(class_path):
        image_files = [f for f in os.listdir(class_path) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
        class_counts.append({"class": class_name, "count": len(image_files)})

count_df = pd.DataFrame(class_counts)
count_df

In [ ]:
plt.figure(figsize=(10, 5))
plt.bar(count_df["class"], count_df["count"])
plt.title("Jumlah Gambar per Class")
plt.xlabel("Class")
plt.ylabel("Jumlah Gambar")
plt.xticks(rotation=30)
plt.show()

In [ ]:
plt.figure(figsize=(12, 8))

sample_index = 1
for class_name in sorted(os.listdir(dataset_dir)):
    class_path = os.path.join(dataset_dir, class_name)
    if os.path.isdir(class_path):
        image_files = [f for f in os.listdir(class_path) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
        if image_files:
            img_path = os.path.join(class_path, image_files[0])
            img = image.load_img(img_path, target_size=(160, 160))
            plt.subplot(3, 3, sample_index)
            plt.imshow(img)
            plt.title(class_name)
            plt.axis("off")
            sample_index += 1

plt.tight_layout()
plt.show()

## BAB 5. Preprocessing dan Augmentasi Data

Model MobileNetV2 membutuhkan gambar ukuran `224 x 224`.

Data training diberi augmentasi agar model lebih kuat mengenali variasi gambar.

Data validation hanya dinormalisasi agar evaluasi lebih stabil.

In [ ]:
IMG_SIZE = 224
BATCH_SIZE = 16

train_datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2,
    rotation_range=30,
    zoom_range=0.25,
    width_shift_range=0.15,
    height_shift_range=0.15,
    shear_range=0.15,
    brightness_range=[0.8, 1.2],
    horizontal_flip=True,
    fill_mode="nearest"
)

val_datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2
)

train_generator = train_datagen.flow_from_directory(
    dataset_dir,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    subset="training",
    shuffle=True,
    seed=SEED
)

val_generator = val_datagen.flow_from_directory(
    dataset_dir,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    subset="validation",
    shuffle=False,
    seed=SEED
)

class_names = list(train_generator.class_indices.keys())
num_classes = train_generator.num_classes

print("Class index:", train_generator.class_indices)
print("Jumlah class:", num_classes)
print("Nama class:", class_names)

## BAB 6. Pembuatan Model MobileNetV2

Bagian ini menggunakan Transfer Learning.

MobileNetV2 dipakai sebagai model dasar. Layer awal MobileNetV2 dibekukan terlebih dahulu agar training lebih stabil.

In [ ]:
base_model = MobileNetV2(
    input_shape=(IMG_SIZE, IMG_SIZE, 3),
    include_top=False,
    weights="imagenet"
)

base_model.trainable = False

model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),

    layers.Dense(256, activation="relu"),
    layers.BatchNormalization(),
    layers.Dropout(0.4),

    layers.Dense(128, activation="relu"),
    layers.BatchNormalization(),
    layers.Dropout(0.3),

    layers.Dense(num_classes, activation="softmax")
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()

### Training Tahap 1

Tahap ini melatih layer tambahan di atas MobileNetV2.

In [ ]:
checkpoint = ModelCheckpoint(
    "skin_disease_model.h5",
    monitor="val_accuracy",
    save_best_only=True,
    mode="max",
    verbose=1
)

early_stop = EarlyStopping(
    monitor="val_loss",
    patience=7,
    restore_best_weights=True,
    verbose=1
)

reduce_lr = ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.2,
    patience=3,
    min_lr=0.000001,
    verbose=1
)

history = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=30,
    callbacks=[checkpoint, early_stop, reduce_lr]
)

### Fine Tuning

Fine tuning melatih beberapa layer terakhir MobileNetV2 agar lebih sesuai dengan dataset kulit.

Jalankan bagian ini setelah training tahap 1 selesai.

In [ ]:
base_model.trainable = True

for layer in base_model.layers[:-40]:
    layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.00001),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

fine_tune_history = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=15,
    callbacks=[checkpoint, early_stop, reduce_lr]
)

## BAB 7. Evaluasi Model

Bagian ini mengecek akurasi model pada data validation.

In [ ]:
loss, accuracy = model.evaluate(val_generator)

print("Validation Loss:", round(loss, 4))
print("Validation Accuracy:", round(accuracy * 100, 2), "%")

In [ ]:
def plot_history(history_obj, title_prefix="Training"):
    plt.figure(figsize=(8, 5))
    plt.plot(history_obj.history["accuracy"], label="Training Accuracy")
    plt.plot(history_obj.history["val_accuracy"], label="Validation Accuracy")
    plt.title(f"{title_prefix} Accuracy")
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy")
    plt.legend()
    plt.show()

    plt.figure(figsize=(8, 5))
    plt.plot(history_obj.history["loss"], label="Training Loss")
    plt.plot(history_obj.history["val_loss"], label="Validation Loss")
    plt.title(f"{title_prefix} Loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.legend()
    plt.show()

plot_history(history, "Tahap 1")
plot_history(fine_tune_history, "Fine Tuning")

### Confusion Matrix dan Classification Report

Bagian ini menunjukkan class mana yang sering benar dan class mana yang sering tertukar.

In [ ]:
val_generator.reset()
y_pred_prob = model.predict(val_generator)
y_pred = np.argmax(y_pred_prob, axis=1)
y_true = val_generator.classes

print(classification_report(y_true, y_pred, target_names=class_names))

cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(9, 7))
sns.heatmap(cm, annot=True, fmt="d", xticklabels=class_names, yticklabels=class_names, cmap="Blues")
plt.title("Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()

### Uji Prediksi Satu Gambar

Bagian ini tidak wajib. Gunakan untuk demo prediksi satu gambar.

Upload satu gambar kulit dengan format JPG, JPEG, atau PNG.

In [ ]:
from google.colab import files


def predict_skin_image(img_path):
    img = image.load_img(img_path, target_size=(IMG_SIZE, IMG_SIZE))
    img_array = image.img_to_array(img)
    img_array = img_array / 255.0
    img_array = np.expand_dims(img_array, axis=0)

    prediction = model.predict(img_array)[0]
    top_3_indices = prediction.argsort()[-3:][::-1]

    plt.imshow(image.load_img(img_path))
    plt.axis("off")
    plt.show()

    print("Top 3 Prediksi:")
    for i in top_3_indices:
        print(class_names[i], ":", round(prediction[i] * 100, 2), "%")

    best_index = top_3_indices[0]
    predicted_class = class_names[best_index]
    confidence = prediction[best_index] * 100

    print("
Hasil utama:", predicted_class)
    print("Confidence:", round(confidence, 2), "%")

    if confidence < 60:
        print("
Status: Model belum yakin")
        print("Saran: Gunakan gambar yang lebih jelas, terang, dan fokus pada area kulit.")
    else:
        print("
Status: Prediksi cukup kuat")

    return predicted_class, confidence

uploaded_test = files.upload()

if len(uploaded_test) > 0:
    test_img_path = list(uploaded_test.keys())[0]
    predict_skin_image(test_img_path)
else:
    print("Tidak ada gambar yang diupload.")

## BAB 8. Simpan Model dan Integrasi ke Web

Bagian ini menyimpan model dan nama class.

File yang dihasilkan:

- `skin_disease_model.h5`
- `class_names.txt`

Masukkan dua file ini ke folder web:

`skinvision-ai-web/model/`

In [ ]:
model.save("skin_disease_model.h5")

with open("class_names.txt", "w") as f:
    for name in class_names:
        f.write(name + "
")

print("Model dan class names berhasil disimpan.")
print("File model: skin_disease_model.h5")
print("File label: class_names.txt")
print("Class names:", class_names)

In [ ]:
from google.colab import files

files.download("skin_disease_model.h5")
files.download("class_names.txt")

## Kesimpulan

Notebook ini menghasilkan model klasifikasi gambar kulit berbasis MobileNetV2. Model ini dapat digunakan pada web SkinVision AI untuk menampilkan prediksi kondisi kulit dan tingkat confidence.

Jika akurasi belum mencapai target, perbaikan dapat dilakukan dengan:

- Menambah jumlah gambar pada class yang sedikit.
- Membersihkan gambar yang salah label.
- Menggunakan gambar dengan pencahayaan lebih konsisten.
- Mengurangi class yang terlalu mirip.
- Melakukan fine tuning dengan jumlah layer yang berbeda.